In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :reciprocal

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [reciprocal_model] Fitting chain 3 (tau=34)
[ Info: [reciprocal] iter 1000/1000000 elapsed=3.9s, rate=0.161, mean=[1.518, 0.00116, 1.270, 0.307], std=[0.3066, 0.000361, 0.1308, 0.2156] [ADAPT]
[ Info: [reciprocal] iter 2000/1000000 elapsed=6.9s, rate=0.120, mean=[1.837, 0.00099, 1.379, 0.246], std=[0.3590, 0.000304, 0.1362, 0.1623] [ADAPT]
[ Info: [reciprocal] iter 3000/1000000 elapsed=9.0s, rate=0.109, mean=[1.966, 0.00093, 1.463, 0.230], std=[0.3418, 0.000267, 0.1557, 0.1348] [ADAPT]
[ Info: [reciprocal] iter 4000/1000000 elapsed=11.2s, rate=0.104, mean=[2.056, 0.00090, 1.515, 0.220], std=[0.3275, 0.000242, 0.1571, 0.1179] [ADAPT]
[ Info: [reciprocal] iter 5000/1000000 elapsed=13.4s, rate=0.103, mean=[2.080, 0.00088, 1.544, 0.218], std=[0.2976, 0.000223, 0.1501, 0.1056] [ADAPT]
[ Info: [reciprocal] iter 6000/1000000 elapsed=15.5s, rate=0.100, mean=[2.101, 0.00088, 1.583, 0.211], std=[0.2782, 0.000208, 0.1582, 0.0975] [ADAPT]
[ Info: [reciprocal] iter 7000/1000000 elapsed=17.7